# Transformers and Attention: Does A appear before B?

A small transformer encoder that reads a 6-token sequence and predicts whether
token A comes before token B. Runs on CPU with PyTorch only.

## 1. Setup

**Prediction:** Does `[3, 1, 4, 2, 5, 6]` have A before B? Yes, it decodes to `x A y B z p`.

In [1]:
import copy
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
torch.set_num_threads(2)
print("PyTorch:", torch.__version__)
names = ["unused", "A", "B", "x", "y", "z", "p", "q"]

def decode(row):
    return " ".join(names[int(token)] for token in row)

print(decode(torch.tensor([3, 1, 4, 2, 5, 6])))

PyTorch: 2.15.0
x A y B z p


**Result:** It printed `x A y B z p`, so yes. Token IDs are just labels for categories. ID 2 isn't "twice" ID 1.

## 2. Attention mixture

**Prediction:** By hand: 0.1·[2,0] + 0.7·[0,4] + 0.2·[1,1] = [0.4, 3.0]. The second source contributes most.

In [2]:
weights = torch.tensor([0.1, 0.7, 0.2])
values = torch.tensor([[2., 0.], [0., 4.], [1., 1.]])
mixed = (weights[:, None] * values).sum(dim=0)
print("Weight sum:", weights.sum().item())
print("Mixed information:", mixed.tolist())

Weight sum: 1.0
Mixed information: [0.4000000059604645, 3.0]


**Result:** It printed [0.4, 3.0] (0.4000000059 is float rounding). The weights sum to 1 and the second source (0.7) dominates. In a real model, softmax of query-key scores makes these weights, and training learns the query, key, and value transforms.

## 3. Opposite-label pairs

**Prediction:** Counting can't work because every sequence has exactly one A and one B. Only the order is different.

In [3]:
generator = torch.Generator().manual_seed(42)
length = 6
pairs = []
seen = set()
while len(pairs) < 300:
    row = torch.randint(3, 8, (length,), generator=generator)
    positions = torch.randperm(length, generator=generator)[:2]
    left, right = sorted(positions.tolist())
    row[left], row[right] = 1, 2
    key = tuple(row.tolist())
    if key in seen:
        continue
    seen.add(key)
    partner = row.clone()
    partner[left], partner[right] = 2, 1
    pairs.append(torch.stack([row, partner]))

X_pairs = torch.stack(pairs)
y_pairs = torch.tensor([[1, 0]] * len(pairs))
for row, label in zip(X_pairs[0], y_pairs[0]):
    print(decode(row), "label", label.item())
print("Dataset shape:", X_pairs.shape)

z z y q A B label 1
z z y q B A label 0
Dataset shape: torch.Size([300, 2, 6])


**Result:** Each pair is the same tokens with A and B swapped, with labels 1 and 0. Shape `[300, 2, 6]` = 300 pairs, 2 sequences each, 6 tokens each. The `seen` set stops duplicates.

## 4. Split and baseline

**Prediction:** The baseline should be 0.5, because every split has complete pairs, so classes are exactly 50/50.

In [4]:
order = torch.randperm(len(pairs), generator=generator)

def partition(ids):
    return X_pairs[ids].reshape(-1, length), y_pairs[ids].reshape(-1)

X_train, y_train = partition(order[:180])
X_val, y_val = partition(order[180:240])
X_test, y_test = partition(order[240:])
majority_class = torch.bincount(y_train).argmax()
baseline = (y_val == majority_class).float().mean().item()
print("Sequence counts:", len(X_train), len(X_val), len(X_test))
print("Validation baseline:", baseline)

Sequence counts: 360 120 120
Validation baseline: 0.5


**Result:** 360 / 120 / 120 sequences. Splitting by pair keeps partners together. Always guessing the majority class gets exactly **0.5**.

## 5. Token embedding

**Prediction:** `[2, 6, 16]` means 2 sequences, 6 tokens each, 16 numbers per token.

In [5]:
torch.manual_seed(7)
token_table = nn.Embedding(num_embeddings=8, embedding_dim=16)
sample = X_train[:2]
vectors = token_table(sample)
print("Token IDs:", sample.shape)
print("Token vectors:", vectors.shape)
print("A vector:", token_table(torch.tensor(1)).detach())

Token IDs: torch.Size([2, 6])
Token vectors: torch.Size([2, 6, 16])
A vector: tensor([-1.5827, -0.3246,  1.9264, -0.3300,  0.1984,  0.7821,  1.0391, -0.7245,
        -0.2093, -0.2153, -1.8157, -0.3452, -2.0615,  0.6741, -1.3233, -1.3598])


**Result:** Correct. Each ID looks up a row of 16 trainable numbers. They start random and the numbers have no English meaning. This gives token identity but not location.

## 6. Position embedding

**Prediction:** It stays `[2, 6, 16]` because the vectors are added, not joined together.

In [6]:
position_table = nn.Embedding(length, 16)
positions = torch.arange(length)
with_positions = vectors + position_table(positions)
print("Position vectors:", position_table(positions).shape)
print("Combined vectors:", with_positions.shape)

Position vectors: torch.Size([6, 16])
Combined vectors: torch.Size([2, 6, 16])


**Result:** The `[6, 16]` position vectors broadcast across both sequences and add to the token vectors. Now A in position 0 is different from A in position 4.

## 7. Classifier

**Prediction:** Batch of 4: IDs `[4, 6]` → embeddings `[4, 6, 16]` → encoder `[4, 6, 16]` → mean pool `[4, 16]` → logits `[4, 2]`.

In [7]:
class OrderClassifier(nn.Module):
    def __init__(self, use_positions=True):
        super().__init__()
        self.use_positions = use_positions
        self.token = nn.Embedding(8, 16)
        self.position = nn.Embedding(length, 16)
        self.encoder = nn.TransformerEncoderLayer(
            d_model=16,
            nhead=2,
            dim_feedforward=32,
            dropout=0.0,
            batch_first=True,
        )
        self.output = nn.Linear(16, 2)

    def forward(self, token_ids):
        vectors = self.token(token_ids)
        if self.use_positions:
            positions = torch.arange(token_ids.size(1))
            vectors = vectors + self.position(positions)
        contextual = self.encoder(vectors)
        sequence_vector = contextual.mean(dim=1)
        return self.output(sequence_vector)

torch.manual_seed(10)
initial = OrderClassifier()
print("Output shape:", initial(X_train[:4]).shape)

Output shape: torch.Size([4, 2])


**Result:** Output `[4, 2]`, matching the trace. The encoder layer has 2 attention heads (8 dims each), a feedforward net, residuals, and layer norm. No causal mask is needed because a classifier can see the whole sequence.

## 8. Why position matters

**Prediction:** No. If both partners get the same scores, they get the same class, so one of them has to be wrong.

In [8]:
without_positions = copy.deepcopy(initial)
without_positions.use_positions = False
without_positions.eval()
with torch.inference_mode():
    pair_logits = without_positions(X_val[:2])
print("Partner scores:", pair_logits)
print("Same scores:", torch.allclose(
    pair_logits[0], pair_logits[1], atol=1e-5))

Partner scores: tensor([[0.0007, 0.2367],
        [0.0007, 0.2367]])
Same scores: True


**Result:** Without positions, the partner scores were identical (`Same scores: True`). Swapping A and B just rearranges the token outputs, and mean pooling erases the order. A model like that is stuck at 50% on these pairs.

## 9. Training loop

**Prediction:** Familiar parts: zero_grad, loss, backward, optimizer step, batches, and eval mode for scoring.

In [9]:
def accuracy(model, X, y):
    model.eval()
    with torch.inference_mode():
        predictions = model(X).argmax(dim=1)
    return (predictions == y).float().mean().item()

def train(model):
    shuffle = torch.Generator().manual_seed(99)
    loader = DataLoader(TensorDataset(X_train, y_train),
                        batch_size=32, shuffle=True, generator=shuffle)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
    loss_function = nn.CrossEntropyLoss()
    best_accuracy = -1.0
    best_weights = None
    for epoch in range(50):
        model.train()
        for X_batch, y_batch in loader:
            optimizer.zero_grad()
            loss = loss_function(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
        score = accuracy(model, X_val, y_val)
        if score > best_accuracy:
            best_accuracy = score
            best_weights = copy.deepcopy(model.state_dict())
        if (epoch + 1) % 10 == 0:
            print("Epoch", epoch + 1, "validation", round(score, 3))
    model.load_state_dict(best_weights)
    return model

**Result:** Same loop as earlier chapters. CrossEntropyLoss takes raw logits, so there's no softmax. Both models use the same starting weights and batch order, and the best validation checkpoint is kept.

## 10. Compare validation

**Prediction:** With positions should beat 50%. Without positions should stay near 50%.

In [10]:
full_model = train(copy.deepcopy(initial))
no_position_model = copy.deepcopy(initial)
no_position_model.use_positions = False
no_position_model = train(no_position_model)
print("Validation baseline:", baseline)
print("Without positions:", accuracy(no_position_model, X_val, y_val))
print("With positions:", accuracy(full_model, X_val, y_val))

full_model.eval()
with torch.inference_mode():
    predictions = full_model(X_val).argmax(dim=1)
for i in range(6):
    print(decode(X_val[i]), "actual", y_val[i].item(),
          "predicted", predictions[i].item())

Epoch 10 validation 1.0
Epoch 20 validation 1.0
Epoch 30 validation 1.0
Epoch 40 validation 1.0


Epoch 50 validation 1.0
Epoch 10 validation 0.5
Epoch 20 validation 0.5
Epoch 30 validation 0.5


Epoch 40 validation 0.5
Epoch 50 validation 0.5
Validation baseline: 0.5
Without positions: 0.5
With positions: 1.0
x A x B p x actual 1 predicted 1
x B x A p x actual 0 predicted 0
q A B q q z actual 1 predicted 1
q B A q q z actual 0 predicted 0
A B y q p x actual 1 predicted 1
B A y q p x actual 0 predicted 0


**Result:** **Baseline 0.5, without positions 0.5, with positions 1.0.** The no-position model never moved off 50% in 50 epochs. Example: `x A x B p x` is actual 1, predicted 1, and its partner `x B x A p x` is actual 0, predicted 0. The model uses position to tell partners apart.

## 11. Final test

**Prediction:** Expected: `x A y B z p` → True, `x B y A z p` → False.

In [11]:
print("Final test accuracy:", accuracy(full_model, X_test, y_test))

new_sequences = torch.tensor([[3, 1, 4, 2, 5, 6],
                              [3, 2, 4, 1, 5, 6]])
full_model.eval()
with torch.inference_mode():
    new_predictions = full_model(new_sequences).argmax(dim=1)
for row, prediction in zip(new_sequences, new_predictions):
    print(decode(row), "A before B?", bool(prediction.item()))

Final test accuracy: 1.0
x A y B z p A before B? True
x B y A z p A before B? False


**Result:** **Final test accuracy: 1.0**, run once after all choices were made. Both hand-written sequences were correct. This only shows it learned this 6-token rule, not language. A chatbot is different: it uses a decoder with causal attention to predict the next token, over and over.

## Summary

| | Validation accuracy |
|---|---|
| Baseline (majority class) | 0.50 |
| Without positions | 0.50 |
| With positions | 1.00 |

**Final test accuracy (with positions): 1.00**

**What attention adds:** each token's vector becomes a weighted mix of
information from every token in the sequence, so a token "knows" its context and
not just its own identity.

**Why position matters:** without position embeddings, the encoder plus mean
pooling gives the same scores to a sequence and its A/B-swapped partner, so it
can't do better than 50%. Position embeddings let the model tell where A and B are.

**Limitation:** this model only knows 8 tokens and length-6 sequences. It
can't handle longer sequences (the position table only has 6 rows) or real
text. The task is also easy enough that a one-line Python check solves it exactly.